# ParlayAPI betting agent starter

Live moneylines in, vig stripped out, one clearly marked hole where your model goes.

This notebook mirrors [`main.py`](https://github.com/JacobiusMakes/parlayapi-betting-agent-starter/blob/main/main.py) in the repo, cell by cell. It runs keyless by default against the [ParlayAPI](https://parlay-api.com) demo endpoints (first 5 events, moneyline only, 60 requests/hour per IP). Set `PARLAY_API_KEY` in the environment for the full feed: the free tier is 1,000 credits/month, no card ([signup](https://parlay-api.com/signup)).

In [1]:
%pip install -q "parlayapi-agent-tools @ git+https://github.com/JacobiusMakes/parlayapi-agent-tools"

In [2]:
import os
import sys

from parlayapi_tools.core import Event, ParlayAPIClient, ParlayAPIError

SPORT = os.environ.get("STARTER_SPORT", "americanfootball_nfl")
MIN_BOOKS = 3  # skip sides priced at fewer books than this: too thin to trust

In [3]:
def american_to_prob(price: float) -> float:
    """Implied win probability of an American price, vig still included."""
    if price >= 100:
        return 100.0 / (price + 100.0)
    return -price / (-price + 100.0)


def prob_to_american(prob: float) -> int:
    """Fair American price for a probability."""
    if prob >= 0.5:
        return round(-100.0 * prob / (1.0 - prob))
    return round(100.0 * (1.0 - prob) / prob)

In [4]:
def fetch_events(client: ParlayAPIClient) -> list[Event]:
    """Keyed feed when PARLAY_API_KEY is set, keyless demo sample otherwise."""
    if client.api_key:
        return client.get_odds(SPORT, markets="h2h", odds_format="american")
    demo = client.demo_odds(SPORT)
    print(f"[demo] Keyless sample: first 5 events, moneyline only, "
          f"{demo.demo_remaining_hour} requests left this hour.")
    print(f"[demo] Set PARLAY_API_KEY for every event and market: "
          f"{demo.demo_signup_url}\n")
    return list(demo.events)

In [5]:
def no_vig_consensus(event: Event) -> dict[str, float]:
    """Consensus fair probability per side of the moneyline.

    Each book's implied probabilities are normalized to sum to 1 (that
    removes the vig book by book), then averaged across books. Simple,
    standard, and a sane baseline for any model to beat.
    """
    per_side: dict[str, list[float]] = {}
    for book in event.bookmakers:
        market = book.market("h2h")
        if market is None:
            continue
        probs = {o.name: american_to_prob(o.price)
                 for o in market.outcomes if o.price is not None}
        overround = sum(probs.values())
        if len(probs) < 2 or overround <= 0:
            continue
        for side, p in probs.items():
            per_side.setdefault(side, []).append(p / overround)
    return {side: sum(ps) / len(ps)
            for side, ps in per_side.items() if len(ps) >= MIN_BOOKS}

## The extension point

`your_model()` is the only function meant to be replaced. It gets one event plus the market's no-vig consensus, and returns your probability per side. The default hands the consensus straight back, so any edge in the table below comes purely from line shopping.

In [6]:
# ---------------------------------------------------------------------------
# >>> YOUR MODEL GOES HERE <<<
#
# This is the extension point, and the only function meant to be replaced.
# It receives one event plus the market's no-vig consensus and returns your
# probability per side. The starter's default just hands the consensus
# back, so any edge you see below comes purely from line shopping (a best
# price that beats the de-vigged market average).
#
# Ideas: an Elo or power rating, a Poisson goals model, injury news, or an
# LLM agent built on these same tools (see the README's agent-tools link).
# ---------------------------------------------------------------------------
def your_model(event: Event, market_fair: dict[str, float]) -> dict[str, float]:
    return market_fair

In [7]:
def build_rows(events: list[Event]) -> list[tuple[str, str, str, str, str, float]]:
    rows = []
    for event in events:
        fair = no_vig_consensus(event)
        model = your_model(event, fair)
        for side in sorted(fair, key=fair.get, reverse=True):
            best = event.best_price("h2h", side)
            if best is None or side not in model:
                continue
            book, price = best
            edge = (model[side] - american_to_prob(price)) * 100.0
            rows.append((event.matchup, side, f"{prob_to_american(fair[side]):+d}",
                         f"{price:+.0f} ({book})", f"{model[side]:.1%}", edge))
    return rows


def print_table(rows: list[tuple[str, str, str, str, str, float]]) -> None:
    header = ("MATCHUP", "SIDE", "FAIR", "BEST PRICE (BOOK)", "MODEL P", "EDGE")
    widths = [max(len(str(r[i])) for r in rows + [header]) for i in range(5)]
    fmt = "  ".join(f"{{:<{w}}}" for w in widths) + "  {:>6}"
    print(fmt.format(*header))
    print(fmt.format(*("-" * w for w in widths), "-" * 6))
    for row in rows:
        print(fmt.format(*row[:5], f"{row[5]:+.1f}%"))

In [8]:
client = ParlayAPIClient()  # reads PARLAY_API_KEY if present
mode = "full API" if client.api_key else "keyless demo"
print(f"parlayapi-betting-agent-starter | sport={SPORT} | mode={mode}\n")

events = fetch_events(client)
rows = build_rows(events)
print_table(rows)

parlayapi-betting-agent-starter | sport=americanfootball_nfl | mode=keyless demo



[demo] Keyless sample: first 5 events, moneyline only, 54 requests left this hour.
[demo] Set PARLAY_API_KEY for every event and market: https://parlay-api.com/signup

MATCHUP                                     SIDE                  FAIR  BEST PRICE (BOOK)  MODEL P    EDGE
------------------------------------------  --------------------  ----  -----------------  -------  ------
New England Patriots at Seattle Seahawks    Seattle Seahawks      -172  -172 (prophetx)    63.2%     +0.0%
New England Patriots at Seattle Seahawks    New England Patriots  +172  +170 (novig)       36.8%     -0.3%
San Francisco 49ers at Los Angeles Rams     Los Angeles Rams      -176  -155 (hardrock)    63.8%     +3.0%
San Francisco 49ers at Los Angeles Rams     San Francisco 49ers   +176  +186 (novig)       36.2%     +1.2%
Chicago Bears at Carolina Panthers          Chicago Bears         -138  -140 (hardrock)    57.9%     -0.4%
Chicago Bears at Carolina Panthers          Carolina Panthers     +138  +141 (novig

## Where to go next

- Swap `your_model()` for a real one: Elo, Poisson, or an LLM agent built on [parlayapi-agent-tools](https://github.com/JacobiusMakes/parlayapi-agent-tools) (LangChain, LlamaIndex, and raw function-calling schemas).
- More worked notebooks: [parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks) (no-vig and EV, line movement, closing line value, parlay pricing).
- [API docs](https://parlay-api.com/docs), [hosted MCP server](https://parlay-api.com/mcp), [pricing](https://parlay-api.com/pricing).